# Práctica 2: Subset Sum — Clase **NP-completo**
### Complejidad Computacional · Facultad de Ciencias, UNAM

---

## 1. Contexto teórico

El problema **Subset Sum** es uno de los 21 problemas originales NP-completos
de Karp (1972).

> **SUBSET-SUM:** Dado un conjunto $\{a_1, a_2, \ldots, a_n\}$ de enteros
> positivos y un entero objetivo $t$, ¿existe un subconjunto
> $S \subseteq \{1, \ldots, n\}$ tal que $\sum_{i \in S} a_i = t$?

**Clases de complejidad relevantes:**
- El problema es **NP-completo** (Karp, 1972).
- El algoritmo de fuerza bruta tarda $O(2^n)$ (exponencial).
- El algoritmo de programación dinámica tarda $O(n \cdot t)$
  (**pseudo-polinomial** — no es polinomial porque $t$ puede ser exponencial
  en el número de bits de la entrada).

**La diferencia entre fuerza bruta y DP** ilustra por qué
"pseudo-polinomial ≠ polinomial": la DP es mucho más rápida cuando $t$ es
pequeño, pero sigue siendo intrateable para $t$ grande.

---

## 2. Objetivo

1. Implementar fuerza bruta con **máscaras de bits** ($O(2^n)$).
2. Implementar la solución de **programación dinámica** ($O(n \cdot t)$).
3. Comparar los tiempos de ejecución de ambos algoritmos.
4. Verificar el comportamiento exponencial empíricamente.

## 3. Mini-tutorial de Python

Conceptos específicos que usaremos en este notebook:

```python
# --- Máscaras de bits ---
# Una "máscara" de n bits representa un subconjunto de {0, 1, ..., n-1}.
# El bit i de `mask` está encendido si (mask >> i) & 1 == 1

n = 4
nums = [3, 1, 4, 1]

mask = 0b0110   # bits 1 y 2 encendidos → subconjunto {nums[1], nums[2]} = {1, 4}

for i in range(n):
    if (mask >> i) & 1:       # bit i encendido?
        print(f"  bit {i} encendido → nums[{i}] = {nums[i]}")

# Para iterar sobre todos los 2^n subconjuntos:
for mask in range(1 << n):    # 1 << n  es lo mismo que  2**n
    print(f"mask = {mask:04b}")

# --- Tablas 2D con listas de listas ---
filas, cols = 3, 5
tabla = [[False] * cols for _ in range(filas)]   # tabla 3×5 de False
tabla[1][3] = True                                # encender posición (1, 3)

# --- Operador OR lógico ---
a = True
b = False
c = a or b    # True  (OR de Python)
```

In [ ]:
# ── Celda 1: Importaciones ────────────────────────────────────────────────────

import time
import random
import matplotlib.pyplot as plt

print("✓ Importaciones listas.")

## 4. Algoritmo 1: Fuerza bruta con máscaras de bits

**Idea:** enumerar los $2^n$ subconjuntos posibles y verificar cuál suma $t$.

Representamos cada subconjunto como un número entero `mask` entre $0$ y $2^n - 1$:
el bit $i$ de `mask` indica si $a_i$ está incluido en el subconjunto.

Por ejemplo, con $n = 3$ y `mask = 5 = 101₂`:
- Bit 0 encendido → $a_0$ incluido
- Bit 1 apagado   → $a_1$ no incluido
- Bit 2 encendido → $a_2$ incluido

In [ ]:
# ── Celda 2: Fuerza bruta con máscaras de bits ───────────────────────────────

def subset_sum_fuerza_bruta(nums, t):
    '''
    Decide si existe un subconjunto de `nums` que sume exactamente `t`.

    Algoritmo: fuerza bruta — prueba los 2^n subconjuntos.
    Complejidad: O(2^n * n)

    Parámetros:
        nums : lista de enteros positivos
        t    : entero objetivo
    Devuelve:
        (True, subconjunto) si existe solución, (False, []) si no.
    '''
    n = len(nums)

    for mask in range(1 << n):   # itera mask = 0, 1, 2, ..., 2^n - 1

        # ╔═══════════════════════════════════════════════════════════════════╗
        # ║  TODO 1 — Calcular la suma del subconjunto representado por mask ║
        # ║                                                                  ║
        # ║  Recuerda: el bit i de mask está encendido si (mask >> i) & 1   ║
        # ║                                                                  ║
        # ║  Instrucciones:                                                  ║
        # ║  1. Inicializa  suma = 0                                         ║
        # ║  2. Para cada i en range(n):                                     ║
        # ║       Si el bit i de mask está encendido:  suma += nums[i]       ║
        # ║  3. (El código que verifica si suma == t ya está escrito abajo)  ║
        # ╚═══════════════════════════════════════════════════════════════════╝

        suma = 0
        for i in range(n):
            pass   # ← REEMPLAZA este pass con: if (mask >> i) & 1: suma += nums[i]

        if suma == t:
            subconj = [nums[i] for i in range(n) if (mask >> i) & 1]
            return True, subconj

    return False, []


# ── Prueba ────────────────────────────────────────────────────────────────────
nums_ej = [3, 34, 4, 12, 5, 2]
t_ej    = 9

encontrado, sub = subset_sum_fuerza_bruta(nums_ej, t_ej)
print(f"nums = {nums_ej},  t = {t_ej}")
print(f"¿Existe subconjunto? {encontrado}")
if encontrado:
    print(f"Subconjunto: {sub}  →  suma = {sum(sub)}")
print()

# Caso sin solución
encontrado2, _ = subset_sum_fuerza_bruta([1, 2, 3], 10)
print(f"¿Existe subconjunto de {{1,2,3}} que sume 10? {encontrado2}  (esperado: False)")

## 5. Algoritmo 2: Programación dinámica (DP)

**Idea:** construir una tabla $dp$ donde $dp[i][s]$ es `True` si se puede
formar la suma $s$ usando los primeros $i$ elementos.

**Transición (relación de recurrencia):**
$$dp[i][s] = dp[i-1][s] \;\lor\; (s \geq a_{i-1} \;\land\; dp[i-1][s - a_{i-1}])$$

En palabras: la suma $s$ es alcanzable con los primeros $i$ elementos si:
- Ya era alcanzable sin el elemento $a_{i-1}$ (**no lo incluimos**), **o bien**
- Era alcanzable la suma $s - a_{i-1}$ con los primeros $i-1$ elementos,
  y luego **incluimos** $a_{i-1}$.

**Casos base:** $dp[0][0] = $ `True` (la suma 0 siempre es posible con 0 elementos).
Todas las demás celdas de la fila 0 son `False`.

In [ ]:
# ── Celda 3: Programación dinámica ───────────────────────────────────────────

def subset_sum_dp(nums, t):
    '''
    Decide si existe un subconjunto de `nums` que sume exactamente `t`.

    Algoritmo: programación dinámica.
    Complejidad: O(n * t)  — pseudo-polinomial

    Parámetros:
        nums : lista de enteros positivos
        t    : entero objetivo
    Devuelve:
        True si existe solución, False si no.
    '''
    n = len(nums)

    # Crear tabla dp de tamaño (n+1) × (t+1), inicializada en False
    # dp[i][s] = ¿se puede formar la suma s con los primeros i elementos?
    dp = [[False] * (t + 1) for _ in range(n + 1)]

    # Caso base: con 0 elementos, la única suma posible es 0
    dp[0][0] = True

    for i in range(1, n + 1):         # i: cuántos elementos consideramos
        for s in range(t + 1):        # s: suma objetivo actual

            # ╔═══════════════════════════════════════════════════════════════╗
            # ║  TODO 2 — Llenar la tabla DP                                ║
            # ║                                                              ║
            # ║  Aplica la transición:                                       ║
            # ║  dp[i][s] = dp[i-1][s]   (no incluir nums[i-1])             ║
            # ║             OR                                               ║
            # ║             (s >= nums[i-1]  AND  dp[i-1][s - nums[i-1]])   ║
            # ║             (incluir nums[i-1], si cabe en la suma s)        ║
            # ║                                                              ║
            # ║  Instrucciones:                                              ║
            # ║  1. Escribe:  dp[i][s] = dp[i-1][s]                         ║
            # ║  2. Si s >= nums[i-1]:                                       ║
            # ║       dp[i][s] = dp[i][s] or dp[i-1][s - nums[i-1]]         ║
            # ╚═══════════════════════════════════════════════════════════════╝

            pass   # ← REEMPLAZA con tu código

    return dp[n][t]


# ── Prueba ────────────────────────────────────────────────────────────────────
print("Pruebas de subset_sum_dp:")
print(f"  [3,34,4,12,5,2], t=9  →  {subset_sum_dp([3,34,4,12,5,2], 9)}   (esperado: True)")
print(f"  [1,2,3],         t=10 →  {subset_sum_dp([1,2,3], 10)}           (esperado: False)")
print(f"  [3,1,1],         t=5  →  {subset_sum_dp([3,1,1], 5)}            (esperado: True)")
print()

# Verificar que coinciden fuerza bruta y DP
for _ in range(200):
    nums_r = random.choices(range(1, 20), k=8)
    t_r    = random.randint(1, sum(nums_r))
    fb, _  = subset_sum_fuerza_bruta(nums_r, t_r)
    dp_    = subset_sum_dp(nums_r, t_r)
    assert fb == dp_, f"¡Discrepancia! nums={nums_r}, t={t_r}"
print("✓ 200 pruebas aleatorias pasaron: fuerza bruta y DP coinciden.")

## 6. Comparación de tiempos

Vamos a comparar el crecimiento del tiempo de ejecución de ambos algoritmos.

- **Fuerza bruta:** mediremos variando $n$ (número de elementos).
- **DP:** mediremos variando $n$ con un $t$ fijo proporcional a $n$.

**Advertencia:** ¡no uses $n$ muy grande con fuerza bruta! Para $n = 30$,
ya son $2^{30} \approx 10^9$ subconjuntos.

In [ ]:
# ── Celda 4: Medición de tiempos — fuerza bruta ──────────────────────────────

tamanos_fb = list(range(5, 26))   # n = 5, 6, ..., 25
tiempos_fb = []
REPS = 3

print(f"{'n':>5}  {'tiempo FB (s)':>16}")
print("-" * 25)

for n in tamanos_fb:
    ts = []
    for rep in range(REPS):
        random.seed(rep * 77 + n)
        nums = random.choices(range(1, 50), k=n)
        t    = random.randint(1, sum(nums) // 2)
        t0   = time.perf_counter()
        subset_sum_fuerza_bruta(nums, t)
        ts.append(time.perf_counter() - t0)
    promedio = sum(ts) / REPS
    tiempos_fb.append(promedio)
    print(f"{n:>5}  {promedio:>16.6f}")

In [ ]:
# ── Celda 5: Medición de tiempos — DP ────────────────────────────────────────

tamanos_dp = list(range(5, 201, 10))   # n = 5, 15, 25, ..., 195
tiempos_dp = []

print(f"{'n':>5}  {'tiempo DP (s)':>16}")
print("-" * 25)

for n in tamanos_dp:
    ts = []
    for rep in range(REPS):
        random.seed(rep * 77 + n)
        nums = random.choices(range(1, 50), k=n)
        t    = 10 * n   # t proporcional a n
        t0   = time.perf_counter()
        subset_sum_dp(nums, t)
        ts.append(time.perf_counter() - t0)
    promedio = sum(ts) / REPS
    tiempos_dp.append(promedio)
    print(f"{n:>5}  {promedio:>16.6f}")

In [ ]:
# ── Celda 6: Gráfica comparativa ─────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Fuerza bruta
axes[0].plot(tamanos_fb, tiempos_fb, "o-", color="crimson", linewidth=2)
axes[0].set_xlabel("n (número de elementos)", fontsize=12)
axes[0].set_ylabel("Tiempo promedio (s)", fontsize=12)
axes[0].set_title("Fuerza bruta — O(2ⁿ)", fontsize=13)
axes[0].grid(True, alpha=0.35)

# DP
axes[1].plot(tamanos_dp, tiempos_dp, "s-", color="seagreen", linewidth=2)
axes[1].set_xlabel("n (número de elementos)", fontsize=12)
axes[1].set_ylabel("Tiempo promedio (s)", fontsize=12)
axes[1].set_title("Programación Dinámica — O(n·t)", fontsize=13)
axes[1].grid(True, alpha=0.35)

plt.suptitle("Práctica 2: Subset Sum — Fuerza Bruta vs. DP", fontsize=14)
plt.tight_layout()
plt.savefig("02_subsetsum_tiempos.png", dpi=110, bbox_inches="tight")
plt.show()
print("Gráfica guardada como  02_subsetsum_tiempos.png")

## 7. Preguntas de análisis

**Q1.** ¿Cómo se ve la curva de la fuerza bruta en comparación con la de DP?
¿A qué función matemática se parece cada una?

**Q2.** La DP tiene complejidad $O(n \cdot t)$. Si en lugar de $t = 10n$
usaras $t = 2^n$, ¿qué pasaría con el tiempo de la DP?
¿Seguiría siendo "más rápida" que la fuerza bruta?

**Q3.** ¿Por qué decimos que $O(n \cdot t)$ es **pseudo-polinomial** y no polinomial?
*(Pista: ¿cuántos bits necesitas para representar el número $t$?)*

**Q4.** Subset Sum es NP-completo. Sin embargo la DP puede resolver instancias
de tamaño $n = 200$ en segundos (con $t = 10n$). ¿Contradice esto que el
problema es NP-completo? Explica.

**Q5.** Menciona una aplicación práctica de Subset Sum (en criptografía,
logística u otro campo).